In [1]:
import torch
import numpy as np
from adapted_sgformer.scripts.train_detection import load_config
from adaptedsgformer.models.detection_models import DetectionGT
from adaptedsgformer.utils import format_data, embed_1D_scalar
from pathlib import Path
from dagr.data.ncaltech101_data import NCaltech101
from torch_geometric.loader import DataLoader
from dagr.data.augment import Augmentations
from argparse import Namespace
from dagr.model.utils import postprocess_network_output, convert_to_training_format
from yolox.models import YOLOX
from torch_scatter import scatter

In [ ]:
TODO:
backbone:
    data = self.events_to_graph(data) --> plus tard

radius: 1.0
uniform_sampling partout

In [2]:
cfg_path = 'config/detection/config_dagt_gnn.yaml'

In [3]:
cfg = load_config(cfg_path)

In [4]:
dataset_path = Path(cfg["data_directory"]) / cfg["dataset"]
augmentations = Augmentations(Namespace(**cfg["augmentations"]))
dataset = NCaltech101(dataset_path, "training", augmentations.transform_training, num_events=cfg["n_nodes"])

In [5]:
len(dataset)

6559

In [6]:
loader = DataLoader(dataset, follow_batch=['bbox', 'bbox0'], shuffle=True, drop_last=True, **cfg["dataloader"])

In [7]:
model = DetectionGT(num_classes=dataset.num_classes, args=cfg["model_params"], height=dataset.height, width=dataset.width).cuda()

poolings: tensor([[0.0179, 0.0250, 1.0000],
        [0.0357, 0.0500, 1.0000],
        [0.0714, 0.1000, 1.0000],
        [0.1429, 0.2000, 1.0000]])
samplings: tensor([2240,  560,  140,   35])


In [8]:
for data in loader:
    data.cuda()
    break

In [9]:
model.eval()

DetectionGT(
  (backbone): BackboneGT(
    (x_embedding): Embedding(2, 24)
    (events_to_graph): EV_TGN()
    (proj): Linear(in_features=36, out_features=36, bias=True)
    (block_dagt): ModuleList(
      (0): BlockDectectGT(
        (pooling): UniformSampling()
        (proj): Linear(in_features=48, out_features=36, bias=True)
        (blockGT): BlockGT(
          (proj): Linear(in_features=36, out_features=48, bias=True)
          (norm1): LayerNorm(36, affine=True, mode=graph)
          (trans): TransLayerMultiHead(
            (Wk): Linear(in_features=36, out_features=48, bias=True)
            (Wq): Linear(in_features=36, out_features=48, bias=True)
            (Wv): Linear(in_features=36, out_features=48, bias=True)
            (Wo): Linear(in_features=48, out_features=48, bias=True)
          )
          (dropout1): Dropout(p=0.1, inplace=False)
          (norm2): LayerNorm(48, affine=True, mode=graph)
          (ff): Sequential(
            (0): Linear(in_features=48, out_feat

In [10]:
data = format_data(data)
with torch.no_grad():
    outputs = model.forward(data)

ICI


In [14]:
len(outputs[0])

8

In [15]:
from yolox.models import YOLOX

In [17]:
outputs = YOLOX.forward(model, data)

In [19]:
outputs.shape

torch.Size([8, 35, 105])

In [15]:
from dagr.model.utils import postprocess_network_output

In [20]:
self = model
detections = postprocess_network_output(outputs, self.head.num_classes, self.conf_threshold, self.nms_threshold, filtering=True,
                                        height=self.height, width=self.width)

RuntimeError: Output 0 of SliceBackward0 is a view and is being modified inplace. This view is the output of a function that returns multiple views. Such functions do not allow the output views to be modified inplace. You should replace the inplace operation by an out-of-place one.

In [10]:
targets = convert_to_training_format(data.bbox, data.bbox_batch, data.num_graphs)

In [11]:
data

DataBatch(x=[373773, 1], y=[8], pos=[373773, 3], p=[8], bbox=[8, 6], bbox_batch=[8], bbox_ptr=[9], t0=[8], t1=[8], width=[8], height=[8], time_window=[8], batch=[373773], ptr=[9])

In [12]:
with torch.no_grad():
    fpn_outs = model.backbone(data)

In [14]:
with torch.no_grad():
    loss, iou_loss, conf_loss, cls_loss, l1_loss, num_fg = model.head(
        fpn_outs, targets, data
    )

In [15]:
loss, iou_loss, conf_loss, cls_loss, l1_loss, num_fg

(tensor(100.7543, device='cuda:0'),
 tensor(4.9840, device='cuda:0'),
 tensor(20.8060, device='cuda:0'),
 tensor(74.9642, device='cuda:0'),
 0.0,
 1.0)

In [ ]:
TODO: postprocess_network_output

In [13]:
fpn_outs

DataBatch(x=[280, 64], pos=[280, 3], width=[8], height=[8], time_window=[8], dist_mat=[280, 35], batch=[280], ptr=[9], edge_index=[2, 3240], edge_attr=[3240, 3])

In [16]:
self = model.head

In [17]:
self.eval()

SparseYoloxHead(
  (stem): ConvBlock(
    (conv): MySplineConv(64, 64, dim=2)
    (norm): BatchNormData(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (cls_conv): ConvBlock(
    (conv): MySplineConv(64, 64, dim=2)
    (norm): BatchNormData(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (cls_pred): MySplineConv(64, 100, dim=2)
  (reg_conv): ConvBlock(
    (conv): MySplineConv(64, 64, dim=2)
    (norm): BatchNormData(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (reg_pred): MySplineConv(64, 4, dim=2)
  (obj_pred): MySplineConv(64, 1, dim=2)
  (l1_loss): L1Loss()
  (bcewithlog_loss): BCEWithLogitsLoss()
  (iou_loss): IOUloss()
)

In [18]:
xin = fpn_outs.clone()

In [19]:
        ev_out = dict(outputs=[], origin_preds=[], x_shifts=[], y_shifts=[], expanded_strides=[])

        batch_size = xin.num_graphs
        normalizer = torch.stack([xin.width[0], xin.height[0]], dim=0)

        #Process data for the first stage of detection 
        cls_output, reg_output, obj_output = self.process_feature(xin)

        self.collect_outputs(cls_output, reg_output, obj_output, self.strides[0], batch_size, normalizer, ret=ev_out)

In [20]:
ev_out

{'outputs': [tensor([[-0.0067,  0.0483, -0.0101,  ...,  0.5033,  0.4942,  0.5050],
          [-0.0207,  0.0302, -0.0403,  ...,  0.5092,  0.4845,  0.5068],
          [ 0.0094,  0.0466, -0.0355,  ...,  0.5135,  0.4857,  0.5138],
          ...,
          [ 0.0427, -0.0896,  0.0390,  ...,  0.5519,  0.4615,  0.4674],
          [ 0.1740, -0.1330,  0.0137,  ...,  0.5651,  0.4642,  0.4710],
          [ 0.3625, -0.1227, -0.0334,  ...,  0.5022,  0.4974,  0.4463]],
         device='cuda:0', grad_fn=<CatBackward0>)],
 'origin_preds': [],
 'x_shifts': [],
 'y_shifts': [],
 'expanded_strides': []}

In [21]:
ev_out['outputs'][0]

tensor([[-0.0067,  0.0483, -0.0101,  ...,  0.5033,  0.4942,  0.5050],
        [-0.0207,  0.0302, -0.0403,  ...,  0.5092,  0.4845,  0.5068],
        [ 0.0094,  0.0466, -0.0355,  ...,  0.5135,  0.4857,  0.5138],
        ...,
        [ 0.0427, -0.0896,  0.0390,  ...,  0.5519,  0.4615,  0.4674],
        [ 0.1740, -0.1330,  0.0137,  ...,  0.5651,  0.4642,  0.4710],
        [ 0.3625, -0.1227, -0.0334,  ...,  0.5022,  0.4974,  0.4463]],
       device='cuda:0', grad_fn=<CatBackward0>)

In [22]:
out = ev_out['outputs']

In [26]:
out[0].shape

torch.Size([280, 105])

In [24]:
            self.hw = [x.shape[-2:] for x in out]
            outputs = torch.cat([x.flatten(start_dim=2) for x in out], dim=2).permute(0, 2, 1)

In [ ]:
x.shape: bs, 105, h, w

In [31]:
outputs = torch.cat([x.view(batch_size, -1, self.num_classes + 5) for x in out], dim=1)

In [29]:
out[0].type()

'torch.cuda.FloatTensor'

In [25]:
hw

[torch.Size([280, 105])]

In [37]:
            output = torch.cat(
                [reg_output.x, obj_output.x.sigmoid(), cls_output.x.sigmoid()], 1
            )
            # output, _ = self.get_output_and_grid(output, cls_output.pos, self.strides[0], batch_size, normalizer)

In [38]:
output.shape

torch.Size([280, 105])

In [40]:
pos =  cls_output.pos

In [42]:
 pos[:, :2]

tensor([[0.1833, 0.5000],
        [0.3125, 0.4667],
        [0.4458, 0.7000],
        [0.5625, 0.3222],
        [0.2917, 0.4556],
        [0.3917, 0.3667],
        [0.1583, 0.5944],
        [0.6375, 0.6278],
        [0.2333, 0.4556],
        [0.3667, 0.4167],
        [0.5333, 0.1167],
        [0.6875, 0.4833],
        [0.3708, 0.7722],
        [0.3500, 0.6722],
        [0.5917, 0.2222],
        [0.3333, 0.2556],
        [0.6000, 0.2333],
        [0.6875, 0.1611],
        [0.3208, 0.3944],
        [0.3917, 0.1889],
        [0.6750, 0.1556],
        [0.4292, 0.7611],
        [0.4208, 0.7722],
        [0.6208, 0.1556],
        [0.6583, 0.5389],
        [0.4708, 0.2278],
        [0.2375, 0.4611],
        [0.0917, 0.5167],
        [0.4917, 0.5056],
        [0.5292, 0.2333],
        [0.4917, 0.3944],
        [0.6583, 0.1944],
        [0.4000, 0.1778],
        [0.7417, 0.0778],
        [0.3000, 0.1444],
        [0.0833, 0.5944],
        [0.3583, 0.8611],
        [0.1958, 0.3722],
        [0.2

In [44]:
normalizer[None]

tensor([[240, 180]], device='cuda:0')

In [43]:
(output[:, :2] + pos[:, :2]) * normalizer[None]

tensor([[ 4.2388e+01,  9.8691e+01],
        [ 7.0036e+01,  8.9445e+01],
        [ 1.0924e+02,  1.3438e+02],
        [ 1.3679e+02,  6.9681e+01],
        [ 7.3787e+01,  8.5333e+01],
        [ 9.8611e+01,  6.6003e+01],
        [ 4.9876e+01,  1.1522e+02],
        [ 1.5416e+02,  1.3135e+02],
        [ 5.9046e+01,  8.3119e+01],
        [ 8.8810e+01,  7.2427e+01],
        [ 1.3839e+02,  3.1611e+01],
        [ 1.7049e+02,  1.1112e+02],
        [ 1.2115e+02,  1.5212e+02],
        [ 1.1626e+02,  1.1408e+02],
        [ 1.5514e+02,  5.3712e+01],
        [ 1.0801e+02,  4.2090e+01],
        [ 1.7060e+02,  3.7850e+01],
        [ 1.8368e+02,  4.3984e+01],
        [ 1.0277e+02,  6.6032e+01],
        [ 1.4396e+02,  2.2600e+01],
        [ 2.0581e+02,  2.7901e+01],
        [ 1.3736e+02,  1.3577e+02],
        [ 1.3264e+02,  1.2190e+02],
        [ 2.1099e+02,  1.0169e+01],
        [ 2.0800e+02,  7.6732e+01],
        [ 1.7792e+02,  2.8295e+01],
        [ 1.0966e+02,  6.8258e+01],
        [ 4.8503e+01,  1.039

In [36]:
output[..., :4]

tensor([[[ 4.2388e+01,  9.8691e+01,  3.5639e+01,  3.7239e+01],
         [ 7.0036e+01,  8.9445e+01,  3.4580e+01,  3.6244e+01],
         [ 1.0924e+02,  1.3438e+02,  3.4744e+01,  3.4476e+01],
         ...,
         [ 1.7670e+02,  1.8653e-01,  3.4677e+01,  3.1542e+01],
         [ 2.4226e+02,  8.0558e+00,  3.1733e+01,  3.2003e+01],
         [ 1.2506e+02,  7.0936e+00,  3.6093e+01,  3.4716e+01]],

        [[ 1.2500e+01,  1.1087e+02,  3.5234e+01,  3.7053e+01],
         [ 7.9581e+01,  1.6113e+02,  3.4662e+01,  3.5729e+01],
         [ 4.6743e+01,  7.4840e+01,  3.6297e+01,  3.7466e+01],
         ...,
         [ 1.2726e+02,  6.4166e+01,  3.1513e+01,  3.0639e+01],
         [ 1.2140e+02,  4.2468e+01,  3.2488e+01,  3.1021e+01],
         [ 7.0122e+01,  2.5107e+01,  3.6121e+01,  3.3481e+01]],

        [[ 8.9318e+01,  8.9898e+01,  3.5312e+01,  3.6806e+01],
         [ 1.4768e+02,  7.5577e+01,  3.5462e+01,  3.7876e+01],
         [ 1.9311e+02,  1.1367e+02,  3.5645e+01,  3.5499e+01],
         ...,
         

In [ ]:
        output[:, :2] = (output[:, :2] + pos[:, :2]) * normalizer[None]
        output[:, 2:4] = torch.exp(output[:, 2:4]) * stride
        output = output.view(batch_size, -1, self.num_classes + 5)

        return output, pos.view(batch_size, -1, 3)

In [32]:
outputs.shape

torch.Size([8, 35, 105])

In [ ]:
cls_output.pos, batch_size, normalizer

In [ ]:
outputs = torch.cat([x.view(batch_size, -1, self.num_classes + 5) for x in out], dim=1)

In [ ]:
        outputs[..., :2] = (outputs[..., :2] + self.grid_cache) * self.stride_cache
        outputs[..., 2:4] = torch.exp(outputs[..., 2:4]) * self.stride_cache

In [36]:
from adaptedsgformer.layers.heads import GNNHead, SparseYoloxHead

from argparse import Namespace

In [39]:
        head_args = dict(
            num_classes=dataset.num_classes,
            strides=model.backbone.strides,
            in_channels=model.backbone.hidden_channels_list[-model.backbone.num_scales:], 
            args=Namespace(**cfg["model_params"]['head'])
        )
        head = GNNHead(**head_args).cuda()

In [41]:
model.backbone.poolings[-1]

tensor([0.1429, 0.2000, 1.0000])

In [44]:
xin = fpn_outs.clone()
xin.pooling = model.backbone.poolings[-1].cuda()
batch_size = xin.num_graphs
cls_output_orig, reg_output_orig, obj_output_orig = head.process_feature(xin, head.stem1, head.cls_conv1, head.reg_conv1,
                                                        head.cls_pred1, head.reg_pred1, head.obj_pred1, batch_size=batch_size, cache=head.cache)

In [46]:
output_orig = torch.cat([reg_output_orig, obj_output_orig, cls_output_orig], 1)

In [47]:
output_orig.shape

torch.Size([8, 105, 5, 7])

In [45]:
cls_output_orig.shape, reg_output_orig.shape, obj_output_orig.shape

(torch.Size([8, 100, 5, 7]),
 torch.Size([8, 4, 5, 7]),
 torch.Size([8, 1, 5, 7]))

In [ ]:
TODO (training):

- anchors (locations) are no longer pixel locations but node positions:
    -> modify get_output_and_grid & get_geometry_constraint (what stride to choose?)
    -> x_shifts and y_shift become node x and node y

- implem -> how to code unif sampling (esp. accumulation of dropped nodes features)

- solve problems w/ current implem -> pe max period & dim, graph & edge creation, edge attributes

(inference)

In [18]:
outputs = torch.cat(ev_out['outputs'], 1)

In [19]:
outputs.shape

torch.Size([8, 35, 105])

In [20]:
labels = targets

In [21]:
x_shifts, y_shifts, expanded_strides, origin_preds = ev_out['x_shifts'], ev_out['y_shifts'], ev_out['expanded_strides'], \
    ev_out['origin_preds']

In [22]:
labels.shape

torch.Size([8, 100, 5])

In [23]:
        bbox_preds = outputs[:, :, :4]  # [batch, n_anchors_all, 4]
        obj_preds = outputs[:, :, 4:5]  # [batch, n_anchors_all, 1]
        cls_preds = outputs[:, :, 5:]  # [batch, n_anchors_all, n_cls]

        # calculate targets
        nlabel = (labels.sum(dim=2) > 0).sum(dim=1)  # number of objects

        total_num_anchors = outputs.shape[1]
        x_shifts = torch.cat(x_shifts, 1)  # [1, n_anchors_all]
        y_shifts = torch.cat(y_shifts, 1)  # [1, n_anchors_all]
        expanded_strides = torch.cat(expanded_strides, 1)

In [90]:
x_shifts

tensor([[0., 1., 2., 3., 4., 5., 6., 0., 1., 2., 3., 4., 5., 6., 0., 1., 2., 3.,
         4., 5., 6., 0., 1., 2., 3., 4., 5., 6., 0., 1., 2., 3., 4., 5., 6.]],
       device='cuda:0')

In [91]:
expanded_strides

tensor([[36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36.,
         36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36.,
         36., 36., 36., 36., 36., 36., 36.]], device='cuda:0')

In [24]:
        cls_targets = []
        reg_targets = []
        l1_targets = []
        obj_targets = []
        fg_masks = []

        num_fg = 0.0
        num_gts = 0.0

        for batch_idx in range(outputs.shape[0]):
            num_gt = int(nlabel[batch_idx])
            num_gts += num_gt
            break

In [25]:
                gt_bboxes_per_image = labels[batch_idx, :num_gt, 1:5]
                gt_classes = labels[batch_idx, :num_gt, 0]
                bboxes_preds_per_image = bbox_preds[batch_idx]

In [26]:
gt_bboxes_per_image

tensor([[ 99.5000,  68.0000, 175.0000, 102.0000]], device='cuda:0')

In [ ]:
(gt_matched_classes,
    fg_mask,
    pred_ious_this_matching,
    matched_gt_inds,
    num_fg_img,
) = self.get_assignments( 

In [28]:
        fg_mask, geometry_relation = self.get_geometry_constraint(
            batch_idx,
            gt_bboxes_per_image,
            expanded_strides,
            x_shifts,
            y_shifts,
        )

In [32]:
expanded_strides.shape

torch.Size([1, 3])

In [33]:
x_shifts[batch_idx] * expanded_strides[0, 0], y_shifts[batch_idx] * expanded_strides[0, 1]

(tensor([101.0000, 124.0000, 129.0000,  96.0000,  53.0000,  83.0000, 119.0000,
         131.0000,  45.0000, 127.0000, 213.0000,  59.0000,  81.0000,   3.0000,
          71.0000, 214.0000, 144.0000,  70.0000,  47.0000, 196.0000, 134.0000,
           1.0000,  26.0000, 129.0000, 127.0000, 131.0000,  57.0000,  98.0000,
         182.0000, 209.0000, 191.0000,   1.0000,  59.0000, 154.0000,  74.0000],
        device='cuda:0'),
 tensor([ 98.0000,  55.0000,  91.0000,  82.0000, 117.0000,  94.0000,   1.0000,
          52.0000, 104.0000, 134.0000, 117.0000,  72.0000,  93.0000,  27.0000,
          92.0000,  94.0000,  70.0000,  73.0000,  72.0000,  92.0000,  63.0000,
          15.0000,  75.0000,  53.0000,  38.0000,  90.0000,  70.0000,  79.0000,
          73.0000,  81.0000, 103.0000,   9.0000, 104.0000,  59.0000,  81.0000],
        device='cuda:0'))

In [29]:
fg_mask

tensor([ True,  True,  True,  True,  True,  True, False,  True, False, False,
        False,  True,  True, False,  True, False,  True,  True,  True, False,
         True, False, False,  True,  True,  True,  True,  True, False, False,
        False, False,  True, False,  True], device='cuda:0')

In [34]:
        bboxes_preds_per_image = bboxes_preds_per_image[fg_mask]
        cls_preds_ = cls_preds[batch_idx][fg_mask]
        obj_preds_ = obj_preds[batch_idx][fg_mask]
        num_in_boxes_anchor = bboxes_preds_per_image.shape[0]

In [36]:
import torch.nn.functional as F
from yolox.utils import bboxes_iou

In [38]:
gt_bboxes_per_image.shape, bboxes_preds_per_image.shape

(torch.Size([1, 4]), torch.Size([21, 4]))

In [37]:
pair_wise_ious = bboxes_iou(gt_bboxes_per_image, bboxes_preds_per_image, False)

In [39]:
pair_wise_ious.shape

torch.Size([1, 21])

In [40]:
pair_wise_ious

tensor([[-0.0000, 0.0639, -0.0000, -0.0000, -0.0000, 0.0372, 0.0057, 0.0488, -0.0000,
         0.0292, -0.0000, 0.0024, 0.0084, 0.0009, 0.0227, 0.0132, -0.0000, 0.0369,
         0.0000, -0.0000, -0.0000]], device='cuda:0', grad_fn=<DivBackward0>)

In [41]:
        gt_cls_per_image = (
            F.one_hot(gt_classes.to(torch.int64), self.num_classes)
            .float()
        )
        pair_wise_ious_loss = -torch.log(pair_wise_ious + 1e-8)

In [43]:
pair_wise_ious_loss

tensor([[18.4207,  2.7497, 18.4207, 18.4207, 18.4207,  3.2906,  5.1756,  3.0201,
         18.4207,  3.5339, 18.4207,  6.0278,  4.7793,  6.9619,  3.7874,  4.3270,
         18.4207,  3.2993, 18.4207, 18.4207, 18.4207]], device='cuda:0',
       grad_fn=<NegBackward0>)

In [45]:
        cost = (
            pair_wise_cls_loss
            + 3.0 * pair_wise_ious_loss
            + float(1e6) * (~geometry_relation)
        )

In [46]:
cost

tensor([[133.3629,  89.0789, 129.1424, 134.1485, 135.2293,  93.6047,  88.1713,
          88.5341, 144.9031,  88.5759, 119.6251, 103.8525, 100.4594,  93.6615,
          74.9293,  72.7361, 142.8628,  94.6912, 145.5163, 147.5691, 151.5320]],
       device='cuda:0', grad_fn=<AddBackward0>)

In [47]:
gt_classes, num_gt, fg_mask

(tensor([2.], device='cuda:0'),
 1,
 tensor([ True,  True,  True,  True,  True,  True, False,  True, False, False,
         False,  True,  True, False,  True, False,  True,  True,  True, False,
          True, False, False,  True,  True,  True,  True,  True, False, False,
         False, False,  True, False,  True], device='cuda:0'))

In [67]:
pair_wise_cls_loss.shape

torch.Size([1, 21])

In [69]:
pair_wise_ious_loss

tensor([[18.4207,  2.7497, 18.4207, 18.4207, 18.4207,  3.2906,  5.1756,  3.0201,
         18.4207,  3.5339, 18.4207,  6.0278,  4.7793,  6.9619,  3.7874,  4.3270,
         18.4207,  3.2993, 18.4207, 18.4207, 18.4207]], device='cuda:0',
       grad_fn=<NegBackward0>)

In [65]:
pair_wise_cls_loss

tensor([[78.1009, 80.8297, 73.8803, 78.8865, 79.9672, 83.7329, 72.6444, 79.4737,
         89.6410, 77.9740, 64.3631, 85.7691, 86.1214, 72.7757, 63.5670, 59.7552,
         87.6007, 84.7932, 90.2543, 92.3071, 96.2700]], device='cuda:0',
       grad_fn=<SumBackward1>)

In [66]:
float(1e6) * (~geometry_relation)

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],
       device='cuda:0')

In [ ]:
            pair_wise_cls_loss
            + 3.0 * pair_wise_ious_loss
            + float(1e6) * (~geometry_relation)

In [48]:
        (
            num_fg,
            gt_matched_classes,
            pred_ious_this_matching,
            matched_gt_inds,
        ) = self.simota_matching(cost, pair_wise_ious, gt_classes, num_gt, fg_mask)

In [53]:
        matching_matrix = torch.zeros_like(cost, dtype=torch.uint8)

        n_candidate_k = min(10, pair_wise_ious.size(1))
        topk_ious, _ = torch.topk(pair_wise_ious, n_candidate_k, dim=1)
        dynamic_ks = torch.clamp(topk_ious.sum(1).int(), min=1)

In [54]:
matching_matrix

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]],
       device='cuda:0', dtype=torch.uint8)

In [55]:
n_candidate_k

10

In [56]:
pair_wise_ious

tensor([[-0.0000, 0.0639, -0.0000, -0.0000, -0.0000, 0.0372, 0.0057, 0.0488, -0.0000,
         0.0292, -0.0000, 0.0024, 0.0084, 0.0009, 0.0227, 0.0132, -0.0000, 0.0369,
         0.0000, -0.0000, -0.0000]], device='cuda:0', grad_fn=<DivBackward0>)

In [57]:
topk_ious

tensor([[0.0639, 0.0488, 0.0372, 0.0369, 0.0292, 0.0227, 0.0132, 0.0084, 0.0057,
         0.0024]], device='cuda:0', grad_fn=<TopkBackward0>)

In [58]:
dynamic_ks

tensor([1], device='cuda:0', dtype=torch.int32)

In [59]:
        for gt_idx in range(num_gt):
            _, pos_idx = torch.topk(
                cost[gt_idx], k=dynamic_ks[gt_idx], largest=False
            )
            matching_matrix[gt_idx][pos_idx] = 1

In [62]:
dynamic_ks[gt_idx]

tensor(1, device='cuda:0', dtype=torch.int32)

In [63]:
cost[gt_idx][pos_idx]

tensor([72.7361], device='cuda:0', grad_fn=<IndexBackward0>)

In [60]:
pos_idx

tensor([15], device='cuda:0')

In [72]:
matching_matrix.shape

torch.Size([1, 21])

In [70]:
anchor_matching_gt = matching_matrix.sum(0)

In [71]:
anchor_matching_gt

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
       device='cuda:0')

In [73]:
        fg_mask_inboxes = anchor_matching_gt > 0
        num_fg = fg_mask_inboxes.sum().item()

In [74]:
fg_mask_inboxes

tensor([False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False,  True, False, False, False, False,
        False], device='cuda:0')

In [75]:
num_fg

1

In [78]:
fg_mask

tensor([False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False,  True, False, False, False, False, False,
        False, False, False, False, False], device='cuda:0')

In [79]:


        matched_gt_inds = matching_matrix[:, fg_mask_inboxes].argmax(0)
        gt_matched_classes = gt_classes[matched_gt_inds]

In [82]:
matching_matrix[:, fg_mask_inboxes]

tensor([[1]], device='cuda:0', dtype=torch.uint8)

In [81]:
matching_matrix

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]],
       device='cuda:0', dtype=torch.uint8)

In [80]:
matched_gt_inds

tensor([0], device='cuda:0')

In [83]:
gt_matched_classes

tensor([2.], device='cuda:0')

In [85]:
pair_wise_ious

tensor([[-0.0000, 0.0639, -0.0000, -0.0000, -0.0000, 0.0372, 0.0057, 0.0488, -0.0000,
         0.0292, -0.0000, 0.0024, 0.0084, 0.0009, 0.0227, 0.0132, -0.0000, 0.0369,
         0.0000, -0.0000, -0.0000]], device='cuda:0', grad_fn=<DivBackward0>)

In [84]:
(matching_matrix * pair_wise_ious).sum(0)[
            fg_mask_inboxes
        ]

tensor([0.0132], device='cuda:0', grad_fn=<IndexBackward0>)

In [49]:
num_fg

1

In [50]:
gt_matched_classes

tensor([2.], device='cuda:0')

In [51]:
pred_ious_this_matching

tensor([0.0132], device='cuda:0', grad_fn=<IndexBackward0>)

In [52]:
matched_gt_inds

tensor([0], device='cuda:0')

In [137]:
        expanded_strides_per_image = expanded_strides[0]
        x_centers_per_image = ((x_shifts[0] + 0.5) * expanded_strides_per_image).unsqueeze(0)
        y_centers_per_image = ((y_shifts[0] + 0.5) * expanded_strides_per_image).unsqueeze(0)

In [138]:
x_centers_per_image

tensor([[ 18.,  54.,  90., 126., 162., 198., 234.,  18.,  54.,  90., 126., 162.,
         198., 234.,  18.,  54.,  90., 126., 162., 198., 234.,  18.,  54.,  90.,
         126., 162., 198., 234.,  18.,  54.,  90., 126., 162., 198., 234.]],
       device='cuda:0')

In [139]:
        center_radius = 1.5
        center_dist = expanded_strides_per_image.unsqueeze(0) * center_radius
        gt_bboxes_per_image_l = (gt_bboxes_per_image[:, 0:1]) - center_dist
        gt_bboxes_per_image_r = (gt_bboxes_per_image[:, 0:1]) + center_dist
        gt_bboxes_per_image_t = (gt_bboxes_per_image[:, 1:2]) - center_dist
        gt_bboxes_per_image_b = (gt_bboxes_per_image[:, 1:2]) + center_dist

In [140]:
center_dist

tensor([[54., 54., 54., 54., 54., 54., 54., 54., 54., 54., 54., 54., 54., 54.,
         54., 54., 54., 54., 54., 54., 54., 54., 54., 54., 54., 54., 54., 54.,
         54., 54., 54., 54., 54., 54., 54.]], device='cuda:0')

In [141]:
gt_bboxes_per_image[:, 0:1]

tensor([[92.5000]], device='cuda:0')

In [144]:
gt_bboxes_per_image_l, gt_bboxes_per_image_r

(tensor([[38.5000, 38.5000, 38.5000, 38.5000, 38.5000, 38.5000, 38.5000, 38.5000,
          38.5000, 38.5000, 38.5000, 38.5000, 38.5000, 38.5000, 38.5000, 38.5000,
          38.5000, 38.5000, 38.5000, 38.5000, 38.5000, 38.5000, 38.5000, 38.5000,
          38.5000, 38.5000, 38.5000, 38.5000, 38.5000, 38.5000, 38.5000, 38.5000,
          38.5000, 38.5000, 38.5000]], device='cuda:0'),
 tensor([[146.5000, 146.5000, 146.5000, 146.5000, 146.5000, 146.5000, 146.5000,
          146.5000, 146.5000, 146.5000, 146.5000, 146.5000, 146.5000, 146.5000,
          146.5000, 146.5000, 146.5000, 146.5000, 146.5000, 146.5000, 146.5000,
          146.5000, 146.5000, 146.5000, 146.5000, 146.5000, 146.5000, 146.5000,
          146.5000, 146.5000, 146.5000, 146.5000, 146.5000, 146.5000, 146.5000]],
        device='cuda:0'))

In [142]:
        c_l = x_centers_per_image - gt_bboxes_per_image_l
        c_r = gt_bboxes_per_image_r - x_centers_per_image
        c_t = y_centers_per_image - gt_bboxes_per_image_t
        c_b = gt_bboxes_per_image_b - y_centers_per_image

In [143]:
c_l, c_r

(tensor([[-20.5000,  15.5000,  51.5000,  87.5000, 123.5000, 159.5000, 195.5000,
          -20.5000,  15.5000,  51.5000,  87.5000, 123.5000, 159.5000, 195.5000,
          -20.5000,  15.5000,  51.5000,  87.5000, 123.5000, 159.5000, 195.5000,
          -20.5000,  15.5000,  51.5000,  87.5000, 123.5000, 159.5000, 195.5000,
          -20.5000,  15.5000,  51.5000,  87.5000, 123.5000, 159.5000, 195.5000]],
        device='cuda:0'),
 tensor([[128.5000,  92.5000,  56.5000,  20.5000, -15.5000, -51.5000, -87.5000,
          128.5000,  92.5000,  56.5000,  20.5000, -15.5000, -51.5000, -87.5000,
          128.5000,  92.5000,  56.5000,  20.5000, -15.5000, -51.5000, -87.5000,
          128.5000,  92.5000,  56.5000,  20.5000, -15.5000, -51.5000, -87.5000,
          128.5000,  92.5000,  56.5000,  20.5000, -15.5000, -51.5000, -87.5000]],
        device='cuda:0'))

In [145]:
        center_deltas = torch.stack([c_l, c_t, c_r, c_b], 2)
        is_in_centers = center_deltas.min(dim=-1).values > 0.0

In [149]:
anchor_filter = is_in_centers.sum(dim=0) > 0

In [151]:
geometry_relation = is_in_centers[:, anchor_filter]

In [152]:
geometry_relation

tensor([[True, True, True, True, True, True, True, True, True]],
       device='cuda:0')

In [148]:
center_deltas

tensor([[[-20.5000, -15.0000, 128.5000, 123.0000],
         [ 15.5000, -15.0000,  92.5000, 123.0000],
         [ 51.5000, -15.0000,  56.5000, 123.0000],
         [ 87.5000, -15.0000,  20.5000, 123.0000],
         [123.5000, -15.0000, -15.5000, 123.0000],
         [159.5000, -15.0000, -51.5000, 123.0000],
         [195.5000, -15.0000, -87.5000, 123.0000],
         [-20.5000,  21.0000, 128.5000,  87.0000],
         [ 15.5000,  21.0000,  92.5000,  87.0000],
         [ 51.5000,  21.0000,  56.5000,  87.0000],
         [ 87.5000,  21.0000,  20.5000,  87.0000],
         [123.5000,  21.0000, -15.5000,  87.0000],
         [159.5000,  21.0000, -51.5000,  87.0000],
         [195.5000,  21.0000, -87.5000,  87.0000],
         [-20.5000,  57.0000, 128.5000,  51.0000],
         [ 15.5000,  57.0000,  92.5000,  51.0000],
         [ 51.5000,  57.0000,  56.5000,  51.0000],
         [ 87.5000,  57.0000,  20.5000,  51.0000],
         [123.5000,  57.0000, -15.5000,  51.0000],
         [159.5000,  57.0000, -

In [146]:
is_in_centers

tensor([[False, False, False, False, False, False, False, False,  True,  True,
          True, False, False, False, False,  True,  True,  True, False, False,
         False, False,  True,  True,  True, False, False, False, False, False,
         False, False, False, False, False]], device='cuda:0')

In [129]:
                    (
                        gt_matched_classes,
                        fg_mask,
                        pred_ious_this_matching,
                        matched_gt_inds,
                        num_fg_img,
                    ) = self.get_assignments(  # noqa
                        batch_idx,
                        num_gt,
                        gt_bboxes_per_image,
                        gt_classes,
                        bboxes_preds_per_image,
                        expanded_strides,
                        x_shifts,
                        y_shifts,
                        cls_preds,
                        obj_preds,
                    )

In [130]:
gt_matched_classes

tensor([15.], device='cuda:0')

In [131]:
fg_mask

tensor([False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False,  True, False, False, False, False, False,
        False, False, False, False, False], device='cuda:0')

In [132]:
pred_ious_this_matching

tensor([0.3398], device='cuda:0')

In [133]:
matched_gt_inds

tensor([0], device='cuda:0')

In [134]:
num_fg_img

1

In [78]:
targets

tensor([[[ 15.0000,  92.5000,  87.0000, 169.0000, 158.0000],
         [  0.0000,   0.0000,   0.0000,   0.0000,   0.0000],
         [  0.0000,   0.0000,   0.0000,   0.0000,   0.0000],
         ...,
         [  0.0000,   0.0000,   0.0000,   0.0000,   0.0000],
         [  0.0000,   0.0000,   0.0000,   0.0000,   0.0000],
         [  0.0000,   0.0000,   0.0000,   0.0000,   0.0000]],

        [[ 11.0000, 107.5000,  83.5000,  93.0000, 117.0000],
         [  0.0000,   0.0000,   0.0000,   0.0000,   0.0000],
         [  0.0000,   0.0000,   0.0000,   0.0000,   0.0000],
         ...,
         [  0.0000,   0.0000,   0.0000,   0.0000,   0.0000],
         [  0.0000,   0.0000,   0.0000,   0.0000,   0.0000],
         [  0.0000,   0.0000,   0.0000,   0.0000,   0.0000]],

        [[  8.0000, 107.0000,  46.0000, 214.0000,  92.0000],
         [  0.0000,   0.0000,   0.0000,   0.0000,   0.0000],
         [  0.0000,   0.0000,   0.0000,   0.0000,   0.0000],
         ...,
         [  0.0000,   0.0000,   0.0000,

In [ ]:
TODO: 
(1) voir cmt la head fonctionne, en part. cmt est construite la ground truth
(2) coder uniform sampling --> modif tous les pooling blocks, bien garder le m format en sortie
(3) modif head

In [40]:
with torch.no_grad():
    model_outputs = model(data)

In [41]:
model_outputs

{'total_loss': tensor(104.3970, device='cuda:0'),
 'iou_loss': tensor(4.9290, device='cuda:0'),
 'l1_loss': 0.0,
 'conf_loss': tensor(25.2903, device='cuda:0'),
 'cls_loss': tensor(74.1776, device='cuda:0'),
 'num_fg': 1.0}